# Searching the PIC-SURE Data Dictionary with Open Access
This tutorial notebook shows how to use the PIC-SURE API with **Open Access**. With Open Access, you do not need to log in with your RAS credentials or have any approved dbGaP studies. All you need is a public access key, which lets you search the PIC-SURE data dictionary.

## PIC-SURE python API
### What is PIC-SURE?
As part of the *NHLBI BioData Catalyst® (BDC)* ecosystem, the Patient Information Commons: Standard Unification of Research Elements (PIC-SURE) platform has been integrating clinical and genomic datasets from multiple TOPMed and TOPMed-related studies funded by the National Heart, Lung, and Blood Institute (NHLBI).

Original data exposed through the PIC-SURE API encompasses a large heterogeneity of data organization underneath. PIC-SURE hides this complexity and exposes the different study datasets in a single tabular format. By simplifying the process of data extraction, it allows investigators to focus on downstream analysis and to facilitate reproducible science.

PIC-SURE API GitHub repositories:
* https://github.com/hms-dbmi/pic-sure-python-adapter-hpds

 ------- 

## Getting your public access key
To use Open Access through the API, you will need a public access key. To request one:

1. In a web browser, go to the [*BDC Powered by PIC-SURE* website](https://picsure.biodatacatalyst.nhlbi.nih.gov/).
2. Navigate to the **API** page.
3. Request a public access key and copy it.

**Keep your key private. Do not share it or commit it to a code repository.**

Next, create a file in your working directory called `open_token.txt`. Paste the public access key in this file and save it. 

## Environment set-up

### Pre-requisites
* python 3.10.20 or later
* pip python package manager, already available in most systems with a python interpreter installed

### Install packages
The first step to using the PIC-SURE API is to install the `picsure` package.

In [ ]:
import sys
import pandas as pd
# BDC Powered by Terra users uncomment the following line to specify package install location
# sys.path.insert(0, r"/home/jupyter/.local/lib/python3.7/site-packages")

In [ ]:
!{sys.executable} -m pip install picsure --upgrade

In [ ]:
import picsure

## Connecting to Open PIC-SURE
The following is required to connect to PIC-SURE with Open Access:
* the platform that should be connected, `picsure.Platform.BDC_OPEN`
* your public access key, read from the `open_token.txt` file created above

Note that we use `picsure.Platform.BDC_OPEN` here rather than `picsure.Platform.BDC_AUTHORIZED`, which is used in the other notebooks in this folder.

In [ ]:
token_file = "open_token.txt"

with open(token_file, "r") as f:
    my_open_token = f.read().strip()

session = picsure.connect(picsure.Platform.BDC_OPEN, my_open_token)

## Searching the PIC-SURE data dictionary
The data dictionary describes every variable available in PIC-SURE. Use `session.searchDictionary()` to search it for a term of interest. The search looks across variable names, descriptions, and study information, and returns the results as a pandas DataFrame.

Let's search for the term `blood pressure`.

In [ ]:
bp_df = session.searchDictionary("blood pressure")  # Search for "blood pressure"
bp_df.shape[0]  # How many variables did the search return?

In [ ]:
bp_df.head()  # View the first 5 rows

Each row of the results is one variable. Some of the most useful columns are:

| Column | Description |
|---|---|
| `conceptPath` | Unique path to the variable; used to build queries with Authorized Access |
| `name` | Variable name (for dbGaP studies, the phv accession) |
| `display` | Short display name of the variable |
| `description` | Description of the variable |
| `dataType` | Whether the variable is categorical or continuous |
| `studyId` | Study accession (e.g. `phs000007`) |
| `studyAcronym` | Study abbreviation (e.g. `FHS`) |
| `values` | For categorical variables, the possible values |
| `min`, `max` | For continuous variables, the minimum and maximum values |

Each variable is organized as a concept path that contains information about the study, variable group, and variable. Though the specifics of the concept paths depend on the type of study, the overall information included is the same.

| Data organization | TOPMed & TOPMed-related studies | BioLINCC & COVID-19 studies (including public data) |
|-------------------|---------------------------------|-----------------------------|
| General organization | Data organized using the format implemented by the database of Genotypes and Phenotypes (dbGaP). Generally, a given study will have several tables, and those tables have several variables. | Data do not follow dbGaP format; there are no phv or pht accessions. Data are organized in groups of like variables, when available. |
| Concept path structure | \\phs\\pht\\phv\\variable name\\ | \\phs\\variable name\\ |
| Variable ID | phv corresponding to the variable accession number | Equivalent to variable name |

Let's take a closer look at the details for the first variable in the search results.

In [ ]:
bp_df.iloc[0] # Look at first variable

We can see which studies these variables come from by counting the results per study.

In [ ]:
bp_df["studyAcronym"].value_counts().head(10)  # Top 10 studies with "blood pressure" variables

## Narrowing a search with facets
As we can see above, results are returned from many different studies. 
Facets let you filter search results by category, such as a specific study or a type of data. These are the same filters shown in the left-hand panel of the PIC-SURE website.

### Viewing the available facets
`session.showAllFacets()` returns every facet category and option as a DataFrame, along with how many variables match each option. The `category` column contains the name to use when selecting a facet, and the `value` column contains the option to select.

In [ ]:
all_facets = session.showAllFacets()
all_facets.category.unique() # What are the facet options?

In [ ]:
all_facets.head(10)

For this example, we will use the **Framingham Heart Study (FHS)**, which has the study accession `phs000007`. 

In [ ]:
study_id = "phs000007"  # Framingham Heart Study

### Searching with a facet
To apply facets to a search:
1. Create a facet object with `session.facets()`.
2. Select options with `.add(category, value)`. You can pass a single value or a list of values.
3. Pass the facet object to `searchDictionary()` with the `facets=` argument.

Let's search for `blood pressure` variables in the Framingham Heart Study only.

In [ ]:
fhs_facets = session.facets()  # Create a facet object with no selections
fhs_facets.add("dataset_id", study_id)  # Select the Framingham Heart Study
fhs_facets.view()  # Show which facets are selected

In [ ]:
fhs_bp_df = session.searchDictionary("blood pressure", facets=fhs_facets)
fhs_bp_df.shape[0]  # How many FHS variables match "blood pressure"?

In [ ]:
fhs_bp_df.head() # See first variables

Studies often record the same concept in several places, such as in different tables, exams, or cohorts. Let's compare how the matching Framingham variables are described and which values they use.

In [ ]:
fhs_bp_df[["name", "display", "description", "dataType", "values"]]

### Clearing facets
To start over, clear all selections with `.clear()`, or clear a single category with `.clear("dataset_id")`.

In [ ]:
fhs_facets.clear()
fhs_facets.view()

### Large searches and paging
Broad search terms can match many thousands of variables. By default, `searchDictionary()` collects every page of results for you. To look at results one page at a time instead, pass `page=` (starting at 0) and `page_size=`. The DataFrame's `attrs` show the total number of matching variables and whether more pages are available.

In [ ]:
first_page = session.searchDictionary("blood pressure", page=0, page_size=100)
first_page.shape[0] # How many variables were returned with this search?

## Saving your search results
Data dictionary results can be saved like any other DataFrame. For example, to save the Framingham `sex` variables to a CSV file:

In [ ]:
fhs_bp_df.to_csv("fhs_bp_variables.csv", index=False)

## Next steps
With Open Access, you have searched the PIC-SURE data dictionary and narrowed the results with facets. To go further:

* **Feasibility assessments with Authorized Access:** To get participant counts and build cohorts with these variables, log in with your eRA Commons account and use your personal access token. See the Authorized Access feasibility notebook *(link to be added; ALS-13754)*.
* **Building queries and retrieving participant-level data:** See [`1_PICSURE_API_101.ipynb`](./1_PICSURE_API_101.ipynb) for how to build queries and export data for studies you are authorized to access.